# Using DataFrames with RLM

This tutorial shows how to pass pandas DataFrames to DSPy's RLM (Recursive Language Model) module while preserving data types.

## Setup

First, install the required packages:

In [ ]:
# !pip install dspy pandas pyarrow

In [ ]:
import dspy
import pandas as pd

# Configure your LM
lm = dspy.LM("openai/gpt-4o-mini")
dspy.configure(lm=lm)

## Create Sample Data

Let's create a DataFrame with various data types to demonstrate type preservation:

In [ ]:
sales_data = pd.DataFrame({
    'date': pd.to_datetime(['2024-01-15', '2024-01-16', '2024-01-17', '2024-01-18', '2024-01-19']),
    'product': pd.Categorical(['Widget', 'Gadget', 'Widget', 'Gizmo', 'Gadget']),
    'quantity': [10, 25, 15, 8, 30],
    'price': [29.99, 49.99, 29.99, 99.99, 49.99],
    'is_promotion': [True, False, True, False, True]
})

print("DataFrame:")
print(sales_data)
print("\nData types:")
print(sales_data.dtypes)

## Define a Signature with DataFrame

Use `dspy.DataFrame` as an input field type. RLM will have full access to the data:

In [ ]:
class AnalyzeSales(dspy.Signature):
    """Analyze sales data and provide insights."""
    
    data: dspy.DataFrame = dspy.InputField(desc="Sales transaction data")
    total_revenue: float = dspy.OutputField(desc="Total revenue across all sales")
    top_product: str = dspy.OutputField(desc="Product with highest quantity sold")
    summary: str = dspy.OutputField(desc="Brief analysis summary")

## Run with RLM

RLM executes Python code in a sandbox where the DataFrame is available with all its types preserved:

In [ ]:
analyzer = dspy.RLM(AnalyzeSales)
result = analyzer(data=sales_data)

print(f"Total Revenue: ${result.total_revenue:.2f}")
print(f"Top Product: {result.top_product}")
print(f"Summary: {result.summary}")

## How It Works

Behind the scenes:

1. **Serialization**: The DataFrame is serialized to Parquet format (via PyArrow), preserving all dtypes
2. **Injection**: The Parquet bytes are sent to RLM's Pyodide sandbox
3. **Execution**: RLM generates and runs Python code that operates on the actual DataFrame
4. **Type Safety**: `datetime64`, `categorical`, `int64`, `float64`, `bool` are all preserved

The LLM sees a preview of the data in the prompt:

In [ ]:
# See what the LLM sees
df_wrapped = dspy.DataFrame(sales_data)
print(df_wrapped.rlm_preview())

## Comparison: RLM vs Other Modules

Only RLM can work with the actual DataFrame. Other modules receive a string representation:

In [ ]:
# This will show a warning - other modules can't use DataFrame directly
import warnings

class SimpleSummary(dspy.Signature):
    """Summarize the data."""
    data: dspy.DataFrame = dspy.InputField()
    summary: str = dspy.OutputField()

# Using Predict instead of RLM - will warn about serialization
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    predictor = dspy.Predict(SimpleSummary)
    # result = predictor(data=sales_data)  # Would trigger warning
    print("Note: dspy.DataFrame works best with dspy.RLM")
    print("Other modules receive JSON-serialized data (types may be lost)")

## Advanced: Custom Types

The same interface (`to_sandbox`, `from_sandbox`, etc.) can be implemented by any `dspy.Type` subclass to enable RLM support for custom data types.